In [1]:
# ============================================================
# relabel_trainval.ipynb
# Tujuan: relabel split_train dan split_val pakai gold labeler
#         yang sama dengan split_test_manual (wordlist + spaCy GPE)
#         supaya evaluasi Model B/C/D konsisten
# ============================================================

# Cell 1 — Install & Import
import pandas as pd
import re
import spacy

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset'

PATH_TRAIN = f'{BASE_PATH}/split_train.csv'
PATH_VAL   = f'{BASE_PATH}/split_val.csv'

df_train = pd.read_csv(PATH_TRAIN)
df_val   = pd.read_csv(PATH_VAL)

print(f'Train : {len(df_train)} baris')
print(f'Val   : {len(df_val)} baris')

Mounted at /content/drive
Train : 574 baris
Val   : 144 baris


In [2]:
# Cell 2 — Definisi make_gold_label (copy persis dari Model A Cell 32)
nlp_geo = spacy.load("en_core_web_sm")

DISEASE_WORDLIST = sorted([
    "nipah virus", "nipah", "niv", "encephalitis", "ensefalitis",
    "meningitis", "respiratory illness", "acute respiratory distress",
    "zoonotic disease", "paramyxovirus", "henipavirus",
    "nipah virus infection", "nipah virus disease"
], key=len, reverse=True)

SYMPTOM_WORDLIST = sorted([
    "fever", "demam", "headache", "sakit kepala", "vomiting", "muntah",
    "seizure", "kejang", "cough", "batuk", "sore throat", "sakit tenggorokan",
    "dizziness", "pusing", "fatigue", "kelelahan", "confusion", "kebingungan",
    "unconsciousness", "tidak sadarkan diri", "respiratory distress",
    "sesak napas", "muscle pain", "nyeri otot", "encephalitis symptoms",
    "drowsiness", "mengantuk", "disorientation", "disorientasi"
], key=len, reverse=True)

def find_exact_terms(text, wordlist):
    found = []
    for term in wordlist:
        pattern = re.compile(r'\b' + re.escape(term) + r'\b', re.IGNORECASE)
        for m in pattern.finditer(text):
            found.append((m.start(), m.end(), m.group()))
    found.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    cleaned = []
    last_end = -1
    for s, e, t in found:
        if s >= last_end:
            cleaned.append(t)
            last_end = e
    return cleaned

def make_gold_label(text):
    gold = {"DISEASE": [], "SYMPTOM": [], "LOCATION": []}
    gold["DISEASE"] = find_exact_terms(text, DISEASE_WORDLIST)
    gold["SYMPTOM"] = find_exact_terms(text, SYMPTOM_WORDLIST)
    doc = nlp_geo(text)
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            gold["LOCATION"].append(ent.text)
    return gold

print("make_gold_label siap!")

make_gold_label siap!


In [3]:
# Cell 3 — Relabel train dan val
def relabel_df(df):
    diseases, symptoms, locations = [], [], []
    for _, row in df.iterrows():
        gold = make_gold_label(str(row['text_clean']))
        diseases.append(', '.join(gold['DISEASE'])  if gold['DISEASE']  else '')
        symptoms.append(', '.join(gold['SYMPTOM'])  if gold['SYMPTOM']  else '')
        locations.append(', '.join(gold['LOCATION']) if gold['LOCATION'] else '')
    df = df.copy()
    df['DISEASE']  = diseases
    df['SYMPTOM']  = symptoms
    df['LOCATION'] = locations
    return df

print("Relabeling train...")
df_train_new = relabel_df(df_train)

print("Relabeling val...")
df_val_new = relabel_df(df_val)

# Verifikasi
for nama, df in [('Train', df_train_new), ('Val', df_val_new)]:
    d = (df['DISEASE']  != '').sum()
    s = (df['SYMPTOM']  != '').sum()
    l = (df['LOCATION'] != '').sum()
    print(f'{nama} → DISEASE:{d}  SYMPTOM:{s}  LOCATION:{l}')

Relabeling train...
Relabeling val...
Train → DISEASE:253  SYMPTOM:26  LOCATION:171
Val → DISEASE:61  SYMPTOM:6  LOCATION:42


In [5]:
# Cell 4 — Simpan (overwrite file lama)
cols = ['id', 'text_clean', 'DISEASE', 'SYMPTOM', 'LOCATION']

df_train_new[cols].to_csv(PATH_TRAIN, index=False)
df_val_new[cols].to_csv(PATH_VAL,   index=False)

print("split_train.csv dan split_val.csv berhasil di-relabel!")
print("Langkah selanjutnya: jalankan ulang iob_conversion.ipynb, lalu retrain Model B")

split_train.csv dan split_val.csv berhasil di-relabel!
Langkah selanjutnya: jalankan ulang iob_conversion.ipynb, lalu retrain Model B
